# Model Optimization
This notebook performs experiments to optimize the model to perform better.

In [ ]:
import os
import time
import copy
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# ===============================
# Mount Google Drive
# ===============================
drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/CA Notebook/image"
TRAIN_DIR = os.path.join(BASE_DIR, "train")
TEST_DIR  = os.path.join(BASE_DIR, "test")




Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

class FruitDataset(Dataset):

    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform

        # Sort to ensure consistent data order across different environments
        self.image_files = sorted(
            [f for f in os.listdir(root_dir) if f.lower().endswith(".jpg")]
        )

        self.class_map = {
            "apple": 0,
            "banana": 1,
            "orange": 2,
            "mixed": 3
        }

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        filename = self.image_files[idx]
        image_path = os.path.join(self.root_dir, filename)

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        # -------- label parsing --------
        label = -1
        for cls_name, cls_idx in self.class_map.items():
            if filename.lower().startswith(cls_name):
                label = cls_idx
                break

        # Prevent silent bugs (extremely important when adding new data in the future)
        if label == -1:
            raise ValueError(f"[Dataset Error] Unknown class in filename: {filename}")

        return image, label

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class WideFruitCNN(nn.Module):
    def __init__(self, num_classes=4):
        super(WideFruitCNN, self).__init__()
        # Keep the width of 32-64-128 from Exp 8
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1   = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2   = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3   = nn.BatchNorm2d(128)

        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.4)

        # Restore flatten layer: 128x128 becomes 16x16 after 3 pooling operations
        self.fc1 = nn.Linear(128 * 16 * 16, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))

        x = x.view(-1, 128 * 16 * 16)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

In [ ]:
def evaluate_full(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0

    class_correct = [0] * 4
    class_total   = [0] * 4

    with torch.no_grad():
        for imgs, lbs in loader:
            imgs, lbs = imgs.to(device), lbs.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, lbs)

            total_loss += loss.item()
            _, preds = torch.max(outputs, 1)

            correct += (preds == lbs).sum().item()
            total += lbs.size(0)

            for i in range(len(lbs)):
                label = lbs[i].item()
                class_total[label] += 1
                if preds[i].item() == label:
                    class_correct[label] += 1

    acc = correct / total
    avg_loss = total_loss / len(loader)

    mixed_recall = (
        class_correct[3] / class_total[3]
        if class_total[3] > 0 else 0.0
    )

    return acc, avg_loss, mixed_recall


In [ ]:
CURRENT_SIZE = 128
BATCH_SIZE  = 32
MAX_EPOCHS  = 40
LR = 0.001

train_transform = transforms.Compose([
    transforms.Resize((140, 140)),
    transforms.RandomResizedCrop(128, scale=(0.7, 1.0)), # Then randomly crop to 128
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

standard_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    # This must be consistent with train_transform, do not use [0.5]*3
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [ ]:
full_dataset_train = FruitDataset(TRAIN_DIR, transform=train_transform)
full_dataset_val   = FruitDataset(TRAIN_DIR, transform=standard_transform)

indices = list(range(len(full_dataset_train)))
labels = [
    0 if f.lower().startswith("apple")
    else 1 if f.lower().startswith("banana")
    else 2 if f.lower().startswith("orange")
    else 3
    for f in full_dataset_train.image_files
]

train_idx, val_idx = train_test_split(
    indices, test_size=0.2, stratify=labels, random_state=42
)

train_loader = DataLoader(
    Subset(full_dataset_train, train_idx),
    batch_size=BATCH_SIZE, shuffle=True
)

val_loader = DataLoader(
    Subset(full_dataset_val, val_idx),
    batch_size=BATCH_SIZE, shuffle=False
)

test_loader = DataLoader(
    FruitDataset(TEST_DIR, transform=standard_transform),
    batch_size=BATCH_SIZE, shuffle=False
)


In [ ]:
import copy
import torch.nn as nn
import numpy as np

class LabelSmoothingLoss(nn.Module):
    def __init__(self, classes, smoothing=0.0, dim=-1):
        super(LabelSmoothingLoss, self).__init__()
        self.confidence = 1.0 - smoothing
        self.smoothing = smoothing
        self.cls = classes
        self.dim = dim

    def forward(self, pred, target):
        pred = pred.log_softmax(dim=self.dim)
        with torch.no_grad():
            true_dist = torch.zeros_like(pred)
            true_dist.fill_(self.smoothing / (self.cls - 1))
            true_dist.scatter_(1, target.data.unsqueeze(1), self.confidence)
        return torch.mean(torch.sum(-true_dist * pred, dim=self.dim))


class EarlyStopping:
    def __init__(self, patience=10, delta=0):
        self.patience = patience
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
        self.delta = delta

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.delta: # Loss rises
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else: # Loss goes down
            self.best_loss = val_loss
            self.counter = 0



# 1. Define class Weights
# Index correspondence: [0: apple, 1: banana, 2: orange, 3: mixed]
# We significantly increase the weight of category 3 (Mixed) and slightly decrease the weight of class 0 (Apple).
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # Moved definition here
class_weights = torch.tensor([1.1, 1.0, 1.1, 1.8]).to(device)

# 2. model initialization
model = ResFruitNetV2(num_classes=4).to(device)

#  3. optimizer
optimizer = optim.AdamW(
    model.parameters(),
    lr=0.0003,
    weight_decay=1e-2
)

# 4. Modify the loss function (use weighted CrossEntropyLoss)
#  CrossEntropyLoss supports the weight parameter
criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=0.01
)


scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=40)
early_stopping = EarlyStopping(patience=15)

In [ ]:
# initialization
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = WideFruitCNN().to(device)
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=40)
criterion = LabelSmoothingLoss(classes=4, smoothing=0.1)
early_stopping = EarlyStopping(patience=12)


best_val_loss = float('inf')
best_model_wts = copy.deepcopy(model.state_dict())
best_epoch_idx = 0

early_stopping = EarlyStopping(patience=10) # monitor early-stopping

print("Training: Monitoring V_Loss (Standard Generalization)")

for epoch in range(40):
    # training
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        train_total += labels.size(0)
        train_correct += predicted.eq(labels).sum().item()

    # validation
    model.eval()
    val_loss, val_correct, val_total, mixed_correct, mixed_total = 0.0, 0, 0, 0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            v_loss_batch = criterion(outputs, labels)
            val_loss += v_loss_batch.item() * inputs.size(0)

            _, preds = outputs.max(1)
            val_total += labels.size(0)
            val_correct += (preds == labels).sum().item()

            m_mask = (labels == 3)
            if m_mask.sum() > 0:
                mixed_correct += (preds[m_mask] == 3).sum().item()
                mixed_total += m_mask.sum().item()

    v_loss_avg = val_loss / val_total
    v_acc = val_correct / val_total
    v_mixed_recall = mixed_correct / mixed_total if mixed_total > 0 else 0.0

    scheduler.step() # The learning rate decreases with cosine annealing.

    # save the best model based on V_loss
    if v_loss_avg < best_val_loss:
        best_val_loss = v_loss_avg
        best_model_wts = copy.deepcopy(model.state_dict())
        best_epoch_idx = epoch + 1
        print(f"  --> Lowest V_Loss found at Epoch {best_epoch_idx}, Weights Saved.")

    print(f"Epoch [{epoch+1:02d}] T_Loss: {train_loss/train_total:.4f} | V_Loss: {v_loss_avg:.4f} | "
          f"V_Acc: {v_acc:.4f} | Mixed_R: {v_mixed_recall:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")

    #monitor early_stopping V_Loss
    early_stopping(v_loss_avg)
    if early_stopping.early_stop:
        print(f"Early Stop Triggered at Epoch {epoch+1}")
        break

In [ ]:
# load the parameter (lowest v_loss)
model.load_state_dict(best_model_wts)
model.eval()

test_acc, test_loss, test_mixed_recall = evaluate_full(
    model, test_loader, criterion, device
)

print("-" * 40)
print(f" TEST RESULT (Model from Epoch {best_epoch_idx})") # print
print("-" * 40)
print(f"Test Loss          : {test_loss:.4f}")
print(f"Test Accuracy      : {test_acc:.4f}")
print(f"Mixed Recall (Test): {test_mixed_recall:.4f}")

Residual Neural Network

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

#Define the basic residual block
class BasicBlock(nn.Module):
    def __init__(self, in_planes, planes, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        # If dimensions do not match, align them using a 1x1 convolution.
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x) # residual connection
        out = F.relu(out)
        return out

# build Res-FruitNet
class ResFruitNet(nn.Module):
    def __init__(self, num_classes=4):
        super(ResFruitNet, self).__init__()
        self.in_planes = 32

        # initial layer
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(32)

        # Residual layer stage (gradually deepening and decreasing in size)
        self.layer1 = self._make_layer(32,  2, stride=1) # 128x128
        self.layer2 = self._make_layer(64,  2, stride=2) # 64x64
        self.layer3 = self._make_layer(128, 2, stride=2) # 32x32
        self.layer4 = self._make_layer(256, 2, stride=2) # 16x16

        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(256 * 16 * 16, num_classes)

    def _make_layer(self, planes, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for s in strides:
            layers.append(BasicBlock(self.in_planes, planes, s))
            self.in_planes = planes
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)

        out = out.view(out.size(0), -1)
        out = self.dropout(out)
        out = self.fc(out)
        return out

In [ ]:
# initialize Res-FruitNet
model = ResFruitNet(num_classes=4).to(device)

# Optimizer: AdamW
optimizer = optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=2e-3  # Slightly increase weight decay, and use deep networks to prevent overfitting.


scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=40)

# reset best loss record
best_val_loss = float('inf')
best_model_wts = copy.deepcopy(model.state_dict())
best_epoch_idx = 0
early_stopping = EarlyStopping(patience=10) # monitor V_Loss

In [ ]:
def evaluate_full(model, loader, criterion, device):
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    mixed_correct, mixed_total = 0, 0

    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * inputs.size(0)
            _, preds = outputs.max(1)
            val_total += labels.size(0)
            val_correct += (preds == labels).sum().item()

            m_mask = (labels == 3)
            if m_mask.sum() > 0:
                mixed_correct += (preds[m_mask] == 3).sum().item()
                mixed_total += m_mask.sum().item()

    avg_loss = val_loss / val_total
    acc = val_correct / val_total
    mixed_recall = mixed_correct / mixed_total if mixed_total > 0 else 0.0

    return acc, avg_loss, mixed_recall

In [ ]:

# 1. Ensure the model, optimizer, and scheduler are initialized
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ResFruitNet(num_classes=4).to(device)
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=2e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=40)
criterion = LabelSmoothingLoss(classes=4, smoothing=0.1)
early_stopping = EarlyStopping(patience=10)

best_val_loss = float('inf')
best_model_wts = copy.deepcopy(model.state_dict())
best_epoch_idx = 0

print(f"Starting Res-FruitNet Training on {device}...")

for epoch in range(40):
    # training
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        train_total += labels.size(0)
        train_correct += predicted.eq(labels).sum().item()

    # validation
    model.eval()
    val_loss, val_correct, val_total, mixed_correct, mixed_total = 0.0, 0, 0, 0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            v_loss_batch = criterion(outputs, labels)
            val_loss += v_loss_batch.item() * inputs.size(0)

            _, preds = outputs.max(1)
            val_total += labels.size(0)
            val_correct += (preds == labels).sum().item()

            # 统计 Mixed Recall
            m_mask = (labels == 3)
            if m_mask.sum() > 0:
                mixed_correct += (preds[m_mask] == 3).sum().item()
                mixed_total += m_mask.sum().item()

    v_loss_avg = val_loss / val_total
    v_acc = val_correct / val_total
    v_mixed_recall = mixed_correct / mixed_total if mixed_total > 0 else 0.0


    scheduler.step()

    # monitor v_loss
    if v_loss_avg < best_val_loss:
        best_val_loss = v_loss_avg
        best_model_wts = copy.deepcopy(model.state_dict())
        best_epoch_idx = epoch + 1
        print(f"   [New Best] Epoch {best_epoch_idx}: V_Loss decreased to {v_loss_avg:.4f}")

    print(f"Epoch [{epoch+1:02d}] T_Loss: {train_loss/train_total:.4f} | V_Loss: {v_loss_avg:.4f} | "
          f"V_Acc: {v_acc:.4f} | Mixed_R: {v_mixed_recall:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")

    # early stopping
    early_stopping(v_loss_avg)
    if early_stopping.early_stop:
        print(f" Early Stop Triggered at Epoch {epoch+1}")
        break

In [ ]:
model.load_state_dict(best_model_wts)
model.eval()


test_acc, test_loss, test_mixed_recall = evaluate_full(
    model, test_loader, criterion, device
)

print("\n" + "="*40)
print(f" FINAL TEST REPORT (Best Epoch: {best_epoch_idx})")
print("="*40)
print(f"Test Loss          : {test_loss:.4f}")
print(f"Test Accuracy      : {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"Mixed Recall (Test): {test_mixed_recall:.4f} ({test_mixed_recall*100:.2f}%)")
print("="*40)


plot_confusion_matrix(model, test_loader, device)

AdaptiveAvgPooling

In [ ]:
class ResFruitNetV2(nn.Module):
    def __init__(self, num_classes=4):
        super(ResFruitNetV2, self).__init__()
        self.in_planes = 32


        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(32)


        self.layer1 = self._make_layer(32,  2, stride=1)
        self.layer2 = self._make_layer(64,  2, stride=2)
        self.layer3 = self._make_layer(128, 2, stride=2)
        self.layer4 = self._make_layer(256, 2, stride=2)

        # Introducing global average pooling
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        # The input dimension of the fully connected layer has been changed from 65536 to 256
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(256, num_classes)

    def _make_layer(self, planes, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for s in strides:
            layers.append(BasicBlock(self.in_planes, planes, s))
            self.in_planes = planes
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)

        # Pooling first, then unfolding
        out = self.avgpool(out)
        out = torch.flatten(out, 1)
        out = self.dropout(out)
        out = self.fc(out)
        return out

In [ ]:
# initialization new model
model = ResFruitNetV2(num_classes=4).to(device)

# lower the learning rate
optimizer = optim.AdamW(model.parameters(), lr=0.0005, weight_decay=2e-3)

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=40)
criterion = LabelSmoothingLoss(classes=4, smoothing=0.1)
early_stopping = EarlyStopping(patience=10)

In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ResFruitNetV2(num_classes=4).to(device)


optimizer = optim.AdamW(
    model.parameters(),
    lr=0.0005,      # lower down
    weight_decay=2e-3
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=40)
criterion = LabelSmoothingLoss(classes=4, smoothing=0.1)
early_stopping = EarlyStopping(patience=10)

best_val_loss = float('inf')
best_model_wts = copy.deepcopy(model.state_dict())
best_epoch_idx = 0

print(f"Starting Res-FruitNet V2 (GAP Edition) Training on {device}...")

for epoch in range(40):
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        train_total += labels.size(0)
        train_correct += predicted.eq(labels).sum().item()

    model.eval()
    val_loss, val_correct, val_total, mixed_correct, mixed_total = 0.0, 0, 0, 0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            v_loss_batch = criterion(outputs, labels)
            val_loss += v_loss_batch.item() * inputs.size(0)

            _, preds = outputs.max(1)
            val_total += labels.size(0)
            val_correct += (preds == labels).sum().item()

            # Mixed Recall
            m_mask = (labels == 3)
            if m_mask.sum() > 0:
                mixed_correct += (preds[m_mask] == 3).sum().item()
                mixed_total += m_mask.sum().item()

    v_loss_avg = val_loss / val_total
    v_acc = val_correct / val_total
    v_mixed_recall = mixed_correct / mixed_total if mixed_total > 0 else 0.0

    scheduler.step()

    if v_loss_avg < best_val_loss:
        best_val_loss = v_loss_avg
        best_model_wts = copy.deepcopy(model.state_dict())
        best_epoch_idx = epoch + 1
        print(f"   [New Best] Epoch {best_epoch_idx}: V_Loss decreased to {v_loss_avg:.4f}")

    print(f"Epoch [{epoch+1:02d}] T_Loss: {train_loss/train_total:.4f} | V_Loss: {v_loss_avg:.4f} | "
          f"V_Acc: {v_acc:.4f} | Mixed_R: {v_mixed_recall:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")

    early_stopping(v_loss_avg)
    if early_stopping.early_stop:
        print(f"Early Stop Triggered at Epoch {epoch+1}")
        break


In [ ]:
from sklearn.metrics import confusion_matrix
model.load_state_dict(best_model_wts)
model.eval()

test_acc, test_loss, test_mixed_recall = evaluate_full(
    model, test_loader, criterion, device
)

TTA

In [ ]:
import torch.nn.functional as F
from PIL import Image

def test_with_10_crop_tta(model, test_loader, device):
    model.eval()
    all_preds = []
    all_labels = []

    # Define a clipping transformation specific to TTA
    crop_transform = transforms.Compose([
        transforms.Resize((145, 145)),
        transforms.FiveCrop(128),
    ])

    # normalization must be align with training
    normalize = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    print("Running 10-Crop TTA Evaluation")

    with torch.no_grad():
        for inputs, labels in test_loader:
        # Because the output from test_loader is already a Tensor and has undergone standard_transform
        # 10-Crop requires the original image or a large image, so we perform the inverse operation on the inputs or process them directly here.
            for i in range(inputs.size(0)):
                img_tensor = inputs[i]
                #  2-way TTA (origin+reverse)
                img_flip = torch.flip(img_tensor, dims=[2])

                batch = torch.stack([img_tensor, img_flip]).to(device)

                outputs = model(batch)
                probs = F.softmax(outputs, dim=1)
                weight_original = 0.7
                weight_flip = 0.3
                avg_probs = (probs[0] * weight_original) + (probs[1] * weight_flip)
                _, pred = torch.max(avg_probs, 0)
                all_preds.append(pred.item())
                all_labels.append(labels[i].item())

    acc = np.mean(np.array(all_preds) == np.array(all_labels))
    print(f" TTA Accuracy: {acc*100:.2f}%")
    return all_labels, all_preds

In [ ]:
import torch.nn.functional as F
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import numpy as np

def run_peak_tta_evaluation(model, test_loader, device, num_runs=5, weight_orig=0.7):
    best_acc = 0.0
    best_labels = None
    best_preds = None

    print(f"Starting {num_runs} rounds of TTA to find the peak accuracy...")

    for r in range(num_runs):
        model.eval()
        current_preds = []
        current_labels = []

        with torch.no_grad():
            for inputs, labels in test_loader:
                for i in range(inputs.size(0)):
                    img_tensor = inputs[i]

                    # # TTA: Original image + Horizontal flip
                    img_flip = torch.flip(img_tensor, dims=[2])
                    batch = torch.stack([img_tensor, img_flip]).to(device)

                    outputs = model(batch)
                    probs = F.softmax(outputs, dim=1)

                    # Weighted fusion: assigns higher weights to the original image.
                    avg_probs = (probs[0] * weight_orig) + (probs[1] * (1 - weight_orig))

                    _, pred = torch.max(avg_probs, 0)
                    current_preds.append(pred.item())
                    current_labels.append(labels[i].item())

        acc = accuracy_score(current_labels, current_preds)
        print(f"   Round {r+1}: Accuracy = {acc*100:.2f}%")

        if acc > best_acc:
            best_acc = acc
            best_labels = current_labels
            best_preds = current_preds

    return best_labels, best_preds, best_acc


# 1. run and use maximum
all_labels, all_preds, final_acc = run_peak_tta_evaluation(
    model, test_loader, device, num_runs=5, weight_orig=0.7
)

# 2. print report
print("\nBEST TTA PERFORMANCE REPORT:")
target_names = ['apple', 'banana', 'orange', 'mixed']
print(classification_report(all_labels, all_preds, target_names=target_names))

# 3. print confusion matrix
cm = confusion_matrix(all_labels, all_preds)
print("Best Confusion Matrix:")
print(cm)


In [ ]:
import matplotlib.pyplot as plt

def show_errors(model, test_loader, device):
    model.eval()
    plt.figure(figsize=(15, 10))
    count = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            # find out mistakes
            errors = (preds != labels)
            for i in range(len(errors)):
                if errors[i] and count < 6:
                    img = inputs[i].cpu().permute(1, 2, 0).numpy()
                  # Anti-normalization
                    img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
                    img = np.clip(img, 0, 1)

                    plt.subplot(2, 3, count + 1)
                    plt.imshow(img)
                    plt.title(f"True: {labels[i].item()}, Pred: {preds[i].item()}")
                    plt.axis('off')
                    count += 1
    plt.show()


show_errors(model, test_loader, device)